# Notebook 05: Importing & Standardizing External Datasets

Various identified dataset types are examined, and specific datasets are used for each of them. Datasets were selected based on defined criteria (including free accessibility, occupational or job title information for mapping, sufficient free text for skill extraction, appropriate dataset size, technical readability, and reproducibility). Importing, light cleaning, and documenting all external datasets. Procedure per dataset:
- Import + read raw data
- Light cleaning/normalization
- Quality checks
- Data understanding (initial exploration)
- Storage in a uniform Unified Document Format in data/interim_external/

**Unified Document Schema (Target Schema):** 
All external sources are converted into a unified document format before the actual skill extraction, ensuring that subsequent steps can be reproduced consistently across all sources. 
Columns:
- `doc_id`: unique document ID (source-stable, ideally with a prefix)
- `source_type`: Document type (e.g., `job_ad`, `cv`, `course`, `linkedin_profile`, `scrape`)
- `source_name`: Fixed source identifier (e.g., `kaggle_linkedin_2023_2024`)
- `job_title_raw`: unprocessed title/headline
- `raw_text`: merged free text, basis for skill extraction
- `language`: language (may be normalized globally via LangDetect later)
- `meta_json`: optional additional attributes as JSON (e.g., location, salary, certifications)
- `timestamp`: optional, if available

In [1]:
#Setup + Paths
from pathlib import Path
import os
import pandas as pd
import xml.etree.ElementTree as ET
import numpy as np
import json
import re

# Dynamically determine the project root, reliably and independently of the working directory
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT:", PROJECT_ROOT)

# Path to external data records
DATA_RAW_EXTERNAL = PROJECT_ROOT / "data" / "raw_external"
DATA_INTERIM_EXTERNAL = PROJECT_ROOT / "data" / "interim_external"

# subfolder standards
DATA_RAW_EXTERNAL_STANDARDS = DATA_RAW_EXTERNAL / "standards"
DATA_INTERIM_EXTERNAL.mkdir(parents=True, exist_ok=True)

print("RAW external (standards):", DATA_RAW_EXTERNAL_STANDARDS)
print("INTERIM external:", DATA_INTERIM_EXTERNAL)

for root, dirs, files in os.walk(DATA_RAW_EXTERNAL_STANDARDS):
    print("In", root, "liegen:", files)

PROJECT_ROOT: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung
RAW external (standards): C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw_external\standards
INTERIM external: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim_external
In C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw_external\standards liegen: ['DKZ_Kompetenzen_gueltig.xml', 'DKZ_Suchworte_Kompetenzen_gueltig.xml', 'linkedin skill.txt']


# 1. Standardized Skill/Job Classifications
Subfolder: standards

## 1.1 BA Competencies
Source: BA Competency Catalog (DKZ_Kompetenzen): https://www.arbeitsagentur.de/institutionen/dkz-downloadportal#Kompetenzen; November 19, 2025

In [2]:
# Import and load the BA competency catalog
path_komp = DATA_RAW_EXTERNAL_STANDARDS / "DKZ_Kompetenzen_gueltig.xml"
print("Lade:", path_komp)

tree = ET.parse(path_komp)
root = tree.getroot()
print("Root-Tag:", root.tag)

# Check the first level
first_children_tags = {child.tag for child in list(root)[:5]}
print("Erste Child-Tags:", first_children_tags)

Lade: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw_external\standards\DKZ_Kompetenzen_gueltig.xml
Root-Tag: kompetenzen
Erste Child-Tags: {'kompetenzgruppe'}


In [3]:
# BA Skills in DataFrame
records = []

# Iterate over ALL competency groups that the children have
for gruppe in root.findall(".//kompetenzgruppe"):
    comps = gruppe.findall("kompetenz")
    if not comps:
        # Skip without competencies
        continue

    group_id = gruppe.get("id")
    group_code = gruppe.get("codenr")
    group_label = gruppe.get("bezeichnung")

    for comp in comps:
        records.append({
            "skill_id": comp.get("id"),
            "code": comp.get("codenr"),
            "label": comp.get("bezeichnung"),
            "group_id": group_id,
            "group_code": group_code,
            "group_label": group_label
        })

df_ba_skills = pd.DataFrame(records)

print("Anzahl Kompetenzen:", len(df_ba_skills))
df_ba_skills.head()

Anzahl Kompetenzen: 9325


,skill_id,code,label,group_id,group_code,group_label
0,60101,K 0001-000,Blumenversand,57343,K 0001,Floristik
1,60102,K 0001-001,Gestecke anfertigen,57343,K 0001,Floristik
2,60103,K 0001-002,Girlanden anfertigen (Floristik),57343,K 0001,Floristik
3,60104,K 0001-003,Hochzeitsfloristik,57343,K 0001,Floristik
4,60105,K 0001-004,Hydrokultur,57343,K 0001,Floristik


Data Understanding: BA Skills

In [4]:
# Overview
df_ba_skills.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9325 entries, 0 to 9324
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   skill_id     9325 non-null   object
 1   code         9325 non-null   object
 2   label        9325 non-null   object
 3   group_id     9325 non-null   object
 4   group_code   9325 non-null   object
 5   group_label  9325 non-null   object
dtypes: object(6)
memory usage: 437.2+ KB


In [5]:
# Examples
df_ba_skills.sample(10, random_state=42)

,skill_id,code,label,group_id,group_code,group_label
1512,61438,K 011002-002,Folienschweißen,59968,K 011002,"Kunststoff-, Kautschukverarbeitung, Vulkanisation"
5982,64401,K 090100-018,Fahrschulwesen,60017,K 090100,"Pädagogik, Erziehung, Ausbildung"
7106,65217,K 100005-032,Tasteninstrumente,60041,K 100005,Musikinstrumente
960,60971,K 010500-005,Kalte Konditorei,57411,K 010500,"Back-, Konditorei- und Süßwaren herstellen"
2771,62459,K 030303-057,Natur- und Landschaftsschutzrecht,59988,K 030303,Rechtsgebiete
4275,126810,K 070106-034,Holzbausoftware SEMA,60001,K 070106,Sonstige Software - technische Anwendungsgebiete
8463,65985,K 130901-008,Werbemittel,60082,K 130901,Medien
3813,138341,K 0700-083,Produktkonfiguratoren,57570,K 0700,EDV-Dienstleistungen
8177,65792,K 130600-012,Wirtschaftsglas,60072,K 130600,Glasprodukte
8241,65838,K 130702-001,BMW,60076,K 130702,Kfz-Marken


### Data Understanding: BA Skills
 
- Scope: 9,325 entries (`df_ba_skills`, 6 columns)
- Structure: Each row describes a BA skill
- Structured, hierarchical skills catalog, serving as an additional skill taxonomy/skill level for use in the profile extension

Columns:
- `skill_id`: Internal ID of the skill (BA)
- `code`: Skill code, e.g., `K 0001-000` (hierarchical structure)
- `label`: Plain-text description of the competency (e.g., flower delivery)
- `group_id`: ID of the parent competency group
- `group_code`: Group code, e.g., `K 0001`
- `group_label`: Name of the group (e.g., floristry)

## 1.2 BA Searchwords

In [6]:
# Import BA searchwords related to competencies
path_suchw = DATA_RAW_EXTERNAL_STANDARDS / "DKZ_Suchworte_Kompetenzen_gueltig.xml"
print("Lade:", path_suchw)

tree_sw = ET.parse(path_suchw)
root_sw = tree_sw.getroot()
print("Root-Tag Suchwörter:", root_sw.tag)

first_children_tags_sw = {child.tag for child in list(root_sw)[:5]}
print("Erste Child-Tags (Suchwörter):", first_children_tags_sw)

Lade: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw_external\standards\DKZ_Suchworte_Kompetenzen_gueltig.xml


Root-Tag Suchwörter: suchworte_kompetenzen
Erste Child-Tags (Suchwörter): {'suchwort'}


In [7]:
# DataFrame
sw_records = []

for sw in root_sw.findall(".//suchwort"):
    sw_records.append({
        "searchword_id": sw.get("id"),
        "code": sw.get("codenr"), # Link to skills via codenr
        "searchword": sw.get("name"),
        "searchword_technical": sw.get("name_technisch"),
        "group": sw.get("suchwortGruppe")
    })

df_ba_searchwords = pd.DataFrame(sw_records)

print("Anzahl Suchwörter:", len(df_ba_searchwords))
df_ba_searchwords.head()

Anzahl Suchwörter: 49321


,searchword_id,code,searchword,searchword_technical,group
0,57328,K 00,"Land-, Forstwirtschaft, Gartenbau",LANDFORSTWIRTSCHAFTGARTENBAU,n
1,57343,K 0001,Floristik,FLORISTIK,n
2,60101,K 0001-000,Blumenversand,BLUMENVERSAND,n
3,60101,K 0001-000,Fleurop,FLEUROP,n
4,60101,K 0001-000,Blumenhandel,BLUMENHANDEL,n


Data Understanding: searchwords

In [8]:
df_ba_searchwords.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49321 entries, 0 to 49320
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   searchword_id         49321 non-null  object
 1   code                  49321 non-null  object
 2   searchword            49321 non-null  object
 3   searchword_technical  49321 non-null  object
 4   group                 49321 non-null  object
dtypes: object(5)
memory usage: 1.9+ MB


In [9]:
df_ba_searchwords.sample(10, random_state=123)

,searchword_id,code,searchword,searchword_technical,group
44814,129388,K 130801-029,Großformatdrucksystem Xerox,GROSSFORMATDRUCKSYSTEMXEROX,n
37192,64936,K 090211-000,Lebensmittelkontrolle,LEBENSMITTELKONTROLLE,n
4961,61207,K 010800-001,Achsenvermessung,ACHSENVERMESSUNG,n
41709,133169,K 100104-034,Mobiler Journalismus,MOBILERJOURNALISMUS,n
17606,138249,K 0610-056,Offshore-Bauwerke und -Anlagen,OFFSHOREBAUWERKEUNDANLAGEN,n
45955,66176,K 1314-010,Packhilfsmittel,PACKHILFSMITTEL,n
46334,66209,K 1400-008,FS 3,FS3,n
5173,76396,K 010800-038,SmartRepair,SMARTREPAIR,n
39923,78387,K 100006-115,Ska,SKA,n
18893,63243,K 070100-022,CAD-Programm ecscad,CADPROGRAMMECSCAD,n


### Data Understanding: BA Searchwords

- Size: 49,321 entries (`df_ba_searchwords`, 5 columns)
- Structure: Each row describes a search word that is assigned to a competency.
- Assessment: List of synonyms and search words for BA competencies (for terminology normalization/synonym mapping)

Columns:
- `searchword_id`: Internal ID of the search word
- `code`: Associated competency code (same format as in the competency catalog, e.g., `K 0001-000`)
- `searchword`: Search word in standard spelling
- `searchword_technical`: “Technical” spelling
- `group`: Internal identifier in the dataset (often `n` in the examples, presumably group type)

In [10]:
# Saving the two BA DataFrames as Parquet
path_komp_parquet = DATA_INTERIM_EXTERNAL / "ba_kompetenzen.parquet"
path_suchw_parquet = DATA_INTERIM_EXTERNAL / "ba_suchwoerter.parquet"

df_ba_skills.to_parquet(path_komp_parquet, index=False)
df_ba_searchwords.to_parquet(path_suchw_parquet, index=False)

print("Gespeichert:")
print("  Kompetenzen ", path_komp_parquet)
print("  Suchwörter ", path_suchw_parquet)

Gespeichert:
  Kompetenzen  C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim_external\ba_kompetenzen.parquet
  Suchwörter  C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim_external\ba_suchwoerter.parquet


### Creation of the standardized table df_ba_total

- Join the two BA tables using the competency code (`code`)
- Merging of competency labels and associated search words/synonyms
- Missing fields are filled in via fallback (e.g., `label` - `searchword`)
- Each row represents a combination of a competency (`code`) and an alias/search word

In [11]:
df_ba_total = (
    df_ba_skills
    .merge(df_ba_searchwords, on="code", how="outer")
    .assign(
        canonical_label=lambda df: df["label"].fillna(df["searchword"]),
        synonym=lambda df: df["searchword"].fillna(df["label"]),
        synonym_tech=lambda df: df["searchword_technical"].fillna("")
    )[
        ["skill_id", "code", "canonical_label", "synonym", "synonym_tech", "group_id", "group_code", "group_label"]
    ]
)

In [12]:
# Save
df_ba_total.to_parquet(DATA_INTERIM_EXTERNAL / "ba_total.parquet", index=False)

In [13]:
# Check
df_ba_total.sample(10, random_state=42)
df_ba_total.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49324 entries, 0 to 49323
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   skill_id         48916 non-null  object
 1   code             49324 non-null  object
 2   canonical_label  49324 non-null  object
 3   synonym          49324 non-null  object
 4   synonym_tech     49324 non-null  object
 5   group_id         48916 non-null  object
 6   group_code       48916 non-null  object
 7   group_label      48916 non-null  object
dtypes: object(8)
memory usage: 3.0+ MB


### BA Competencies + Search Terms = Standardized BA Skills Table

2 BA datasets (competencies + search terms) combined into a mapping table; each competency is assigned all available synonyms, spelling variations, and technical variants. BA data is in German and standardized -> ideal for contexts related to KldB. Not used as a primary source for profile expansion but as a kind of skill database/dictionary, especially for German-language external datasets.

Result: `df_ba_total` (approx. 50,000 rows), saved under `data/interim_external/ba_total.parquet`.

# 2. Job Postings/Job Market Articles

Subfolder: job_ads

## 2.1 Dataset: BA Job Search API (custom dataset, DE, ba_jobs_ads_fulltext.csv)

The dataset was created in Notebook 05b. Source: Job Search API of the Federal Employment Agency (https://jobsuche.api.bund.dev/)

In [14]:
# Path to the file
PATH_JOB_ADS = DATA_RAW_EXTERNAL / "job_ads"
PATH_JOB_ADS.mkdir(parents=True, exist_ok=True)

DATA_INTERIM_EXTERNAL.mkdir(parents=True, exist_ok=True)

print("RAW job_ads:", PATH_JOB_ADS)
print("INTERIM:", DATA_INTERIM_EXTERNAL)

RAW job_ads: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw_external\job_ads
INTERIM: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim_external


In [15]:
# loading the files
path_raw = PATH_JOB_ADS / "ba_job_ads_raw.csv"
path_full = PATH_JOB_ADS / "ba_job_ads_fulltext.csv"

df_raw = pd.read_csv(path_raw, low_memory=False)
df_full = pd.read_csv(path_full, low_memory=False)

print("Rows raw:", len(df_raw))
print("Rows fulltext:", len(df_full))

display(df_raw.head(3))
display(df_full.head(3))

Rows raw: 5902
Rows fulltext: 3505


,refnr,jobtitel,beruf,volltext,volltext_methode,seed,externeUrl
0,12117-28424526-YF-S,Kundenbetreuer Pflege (m/w/d),Kundendienstberater/in,Zum Hauptinhalt springen Für unsere Regionalge...,jsonld,Pflege,https://www.yourfirm.de/job/detail/20251220-28...
1,12781-15960381-KAL-S,Mitarbeiter in der Pflege (m/w/d),Krankenschwester/-pfleger,NaN,http_403,Pflege,https://www.kalaydo.de/jobs/15960381/?utm_id=b...
2,11858-15918928-STA-S,Pflege- oder Medizinpädagog*innen (m/w/d),Pflegepädagoge/-pädagogin,Die Universitätsmedizin Göttingen (UMG) verein...,jsonld,Pflege,https://www.stellenanzeigen.de/job/detail/1591...


,refnr,jobtitel,beruf,volltext,volltext_methode,seed,externeUrl
0,12117-28424526-YF-S,Kundenbetreuer Pflege (m/w/d),Kundendienstberater/in,Zum Hauptinhalt springen Für unsere Regionalge...,jsonld,Pflege,https://www.yourfirm.de/job/detail/20251220-28...
1,11858-15918928-STA-S,Pflege- oder Medizinpädagog*innen (m/w/d),Pflegepädagoge/-pädagogin,Die Universitätsmedizin Göttingen (UMG) verein...,jsonld,Pflege,https://www.stellenanzeigen.de/job/detail/1591...
2,12618-161686-S,Prozessmanager*in Pflege,Pflegewissenschaftler/in,Prozessmanager*in Pflege Job in Oerlinghausen ...,fallback_visible_text,Pflege,https://www.awo-jobs.de/index.php?id=114&uid=1...


Only the full-text record is needed, as it contains only complete data with free-text job descriptions.

Structure check:

In [16]:
expected_cols = {"refnr","jobtitel","beruf","volltext","volltext_methode","seed","externeUrl"}
missing = expected_cols - set(df_raw.columns)
print("Missing cols in raw:", missing)

missing_full = expected_cols - set(df_full.columns)
print("Missing cols in fulltext:", missing_full)

# stats
print("Null volltext (raw):", df_raw["volltext"].isna().sum() if "volltext" in df_raw else None)
print("Null volltext (full):", df_full["volltext"].isna().sum() if "volltext" in df_full else None)

Missing cols in raw: set()
Missing cols in fulltext: set()
Null volltext (raw): 2380
Null volltext (full): 0


Basic text cleaning:
- Normalization of whitespace
- Removal of empty or inappropriate text
- Preparation for the Unified Document Schema

In [17]:
def clean_text_basic(s):
    if pd.isna(s):
        return None
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else None

for col in ["jobtitel", "beruf", "volltext", "seed", "externeUrl", "volltext_methode"]:
    if col in df_full.columns:
        df_full[col] = df_full[col].apply(clean_text_basic)

# Calculate text length
df_full["volltext_len"] = df_full["volltext"].apply(
    lambda x: len(x) if isinstance(x, str) else 0
)

df_full["volltext_len"].describe()

count     3505.000000
mean      2674.759486
std       1657.623687
min        360.000000
25%       1836.000000
50%       2246.000000
75%       3026.000000
max      27839.000000
Name: volltext_len, dtype: float64

Deduplication Check:

In [18]:
print("Anzahl Zeilen:", len(df_full))
print("Eindeutige refnr:", df_full["refnr"].nunique())

dups = df_full[df_full["refnr"].duplicated(keep=False)]
print("Duplikate:", len(dups))

Anzahl Zeilen: 3505
Eindeutige refnr: 3505
Duplikate: 0


Data Understanding:

In [19]:
# Distribution + Key Figures
print("Anzahl Seeds:", df_full["seed"].nunique())
display(df_full["seed"].value_counts().head(15))

display(df_full["volltext_methode"].value_counts(dropna=False))

df_full["volltext_len"].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99])

Anzahl Seeds: 70


seed
Tischler         100
Erzieher         100
Mechatroniker    100
Kita              94
HR                93
Spedition         91
Gesundheits       90
Therapie          89
Service           87
Einzelhandel      85
Schicht           84
Altenpfleger      83
Pflege            82
Buchhaltung       79
Controlling       78
Name: count, dtype: int64

volltext_methode
jsonld                   2909
fallback_visible_text     596
Name: count, dtype: int64

count     3505.000000
mean      2674.759486
std       1657.623687
min        360.000000
10%       1465.000000
25%       1836.000000
50%       2246.000000
75%       3026.000000
90%       4283.600000
95%       5337.400000
99%       7594.520000
max      27839.000000
Name: volltext_len, dtype: float64

Unified Document Schema:

In [20]:
def unify_ba_job_ads(df, source_name="ba_jobsuche_api_2025"):
    def build_meta(row):
        meta = {}
        for k in ["refnr", "beruf", "seed", "externeUrl", "volltext_methode", "volltext_len"]:
            if k in row and pd.notna(row[k]):
                meta[k] = row[k]
        return json.dumps(meta, ensure_ascii=False)

    df_unified = pd.DataFrame({
        "doc_id": df["refnr"].astype(str),
        "source_type": "job_ad",
        "source_name": source_name,
        "job_title_raw": df["jobtitel"].fillna(df["beruf"]),
        "raw_text": df["volltext"],
        "language": "de",  # Job listings in German
        "meta_json": df.apply(build_meta, axis=1)
    })

    df_unified["job_title_raw"] = df_unified["job_title_raw"].apply(clean_text_basic)
    df_unified["raw_text"] = df_unified["raw_text"].apply(clean_text_basic)

    return df_unified[df_unified["raw_text"].notna()].copy()

df_ba_job_ads = unify_ba_job_ads(df_full)
display(df_ba_job_ads.head(3))
print("Anzahl vereinheitlichter Dokumente:", len(df_ba_job_ads))

,doc_id,source_type,source_name,job_title_raw,raw_text,language,meta_json
0,12117-28424526-YF-S,job_ad,ba_jobsuche_api_2025,Kundenbetreuer Pflege (m/w/d),Zum Hauptinhalt springen Für unsere Regionalge...,de,"{""refnr"": ""12117-28424526-YF-S"", ""beruf"": ""Kun..."
1,11858-15918928-STA-S,job_ad,ba_jobsuche_api_2025,Pflege- oder Medizinpädagog*innen (m/w/d),Die Universitätsmedizin Göttingen (UMG) verein...,de,"{""refnr"": ""11858-15918928-STA-S"", ""beruf"": ""Pf..."
2,12618-161686-S,job_ad,ba_jobsuche_api_2025,Prozessmanager*in Pflege,Prozessmanager*in Pflege Job in Oerlinghausen ...,de,"{""refnr"": ""12618-161686-S"", ""beruf"": ""Pflegewi..."


Anzahl vereinheitlichter Dokumente: 3505


In [21]:
# Sanity-Check:
print("Eindeutige doc_id:", df_ba_job_ads["doc_id"].nunique())
print("Leere Jobtitel:", df_ba_job_ads["job_title_raw"].isna().mean())

df_ba_job_ads["raw_len"] = df_ba_job_ads["raw_text"].apply(len)
df_ba_job_ads["raw_len"].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99])

Eindeutige doc_id: 3505
Leere Jobtitel: 0.0


count     3505.000000
mean      2674.759486
std       1657.623687
min        360.000000
10%       1465.000000
25%       1836.000000
50%       2246.000000
75%       3026.000000
90%       4283.600000
95%       5337.400000
99%       7594.520000
max      27839.000000
Name: raw_len, dtype: float64

Save:

In [22]:
out_path = DATA_INTERIM_EXTERNAL / "ba_job_ads_unified.parquet"
df_ba_job_ads.drop(columns=["raw_len"], errors="ignore").to_parquet(out_path, index=False)

print("Gespeichert unter:", out_path)

Gespeichert unter: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim_external\ba_job_ads_unified.parquet


## 2.2 Dataset: Job Posting Data in Germany (Job Posting.csv)

Source: Kaggle Techsalerator - Job Posting Data in Germany: https://www.kaggle.com/datasets/techsalerator/job-posting-data-in-germany?select=Job+Posting.csv; December 23, 2025

In [23]:
# Import
path_techs = PATH_JOB_ADS / "Job Posting.csv"

df_raw = pd.read_csv(path_techs, low_memory=False, encoding="latin1") # Import mit encoding="latin1" um Sonderzeichen korrekt zu laden
display(df_raw.head(3))
print("Rows:", len(df_raw), "Cols:", df_raw.shape[1])

,Website Domain,Ticker,Job Opening Title,Job Opening URL,First Seen At,Last Seen At,Location,Location Data,Category,Seniority,...,Description,Salary,Salary Data,Contract Types,Job Status,Job Language,Job Last Processed At,O*NET Code,O*NET Family,O*NET Occupation Name
0,bosch.com,NaN,IN_RBAI_Assistant Manager_Dispensing Process E...,https://jobs.smartrecruiters.com/BoschGroup/74...,2024-05-29T19:59:45Z,2024-07-31T14:35:44Z,"Indiana, United States","[{""city"":null,""state"":""Indiana"",""zip_code"":nul...","engineering, management, support",manager,...,**IN\_RBAI\_Assistant Manager\_Dispensing Proc...,NaN,"{""salary_low"":null,""salary_high"":null,""salary_...",full time,closed,en,2024-08-02T14:47:55Z,43-1011.00,Office and Administrative Support,First-Line Supervisors of Office and Administr...
1,bosch.com,NaN,Professional Internship: Hardware Development ...,https://jobs.smartrecruiters.com/BoschGroup/74...,2024-05-04T01:00:12Z,2024-07-29T17:46:16Z,"Delaware, United States","[{""city"":null,""state"":""Delaware"",""zip_code"":nu...",internship,non_manager,...,**Professional Internship: Hardware Developmen...,NaN,"{""salary_low"":null,""salary_high"":null,""salary_...","full time, internship, m/f",closed,en,2024-07-31T17:50:07Z,17-2061.00,Architecture and Engineering,Computer Hardware Engineers
2,zf.com,NaN,Process Expert BMS Production,https://jobs.zf.com/job/Shenyang-Process-Exper...,2024-04-19T06:47:24Z,2024-05-16T02:25:08Z,China,"[{""city"":null,""state"":null,""zip_code"":null,""co...",engineering,non_manager,...,ZF is a global technology company supplying sy...,NaN,"{""salary_low"":null,""salary_high"":null,""salary_...",NaN,closed,en,2024-05-18T02:32:04Z,51-9141.00,Production,Semiconductor Processing Technicians


Rows: 9919 Cols: 21


9,919 entries with many different columns. Although the companies are German, they are located worldwide. The O*NET (U.S. national occupational classification) code is provided, so we can assume the data has already been processed.

Light data cleaning:
- Normalize whitespace
- Filter out/flag empty descriptions
- Set a minimum length for descriptions

In [24]:
def clean_text_basic(s):
    if pd.isna(s):
        return None
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else None

for col in ["Job Opening Title", "Description", "Location", "Category", "Seniority", "Keywords", "Job Language"]:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].apply(clean_text_basic)

df_raw["desc_len"] = df_raw["Description"].apply(lambda x: len(x) if isinstance(x,str) else 0)
df_raw["desc_len"].describe()

count    9919.000000
mean     3245.976308
std      1436.009861
min         0.000000
25%      2321.000000
50%      3040.000000
75%      4031.000000
max      7980.000000
Name: desc_len, dtype: float64

- clean_text_basic(): Whitespace normalization, null handling
- Length analysis (desc_len.describe()): Median 3.8k characters, max 7.9k characters

Language handling, as entries are partly in German, mostly in English, and some even in the local language. The existing “Job Language” column will be standardized (e.g., `de`, `en`, `other`).

In [25]:
def normalize_lang(x):
    if not isinstance(x, str) or not x.strip():
        return "unknown"
    v = x.strip().lower()
    if v in ["de", "deutsch", "german", "ger"]:
        return "de"
    if v in ["en", "englisch", "english"]:
        return "en"
    return v[:10]  # or other

Unified Document Schema:

In [26]:
def unify_techsalerator(df, source_name="kaggle_techsalerator"):
    def build_meta(row):
        meta = {}
        for k in [
            "Website Domain","Ticker","Job Opening URL","First Seen At","Last Seen At","Location Data","Salary","Salary Data","Contract Types","Job Status",
            "Job Last Processed At","O*NET Code","O*NET Family","O*NET Occupation Name","Category","Seniority","Keywords","Job Language"
        ]:
            v = row.get(k, None)
            if pd.notna(v) and v is not None and str(v).strip() != "":
                meta[k] = v
        return json.dumps(meta, ensure_ascii=False)

    # doc_id: URL if available; otherwise, fallback = row index, as a series since there is no scalar parameter
    fallback_id = pd.Series(df.index.astype(str), index=df.index)
    doc_id_series = df["Job Opening URL"].astype("string").fillna(fallback_id).astype(str)

    df_u = pd.DataFrame({
        "doc_id": doc_id_series,
        "source_type": "job_ad",
        "source_name": source_name,
        "job_title_raw": df["Job Opening Title"].fillna(""),
        "raw_text": df["Description"].fillna(""),
        "language": df["Job Language"].apply(normalize_lang) if "Job Language" in df.columns else "unknown",
        "meta_json": df.apply(build_meta, axis=1),
    })

    # clean + drop empty
    df_u["job_title_raw"] = df_u["job_title_raw"].apply(clean_text_basic)
    df_u["raw_text"] = df_u["raw_text"].apply(clean_text_basic)
    return df_u[df_u["raw_text"].notna()].copy()

# Example Output
df_techsalerator_u = unify_techsalerator(df_raw)
display(df_techsalerator_u.head(3))
df_techsalerator_u.info()

,doc_id,source_type,source_name,job_title_raw,raw_text,language,meta_json
0,https://jobs.smartrecruiters.com/BoschGroup/74...,job_ad,kaggle_techsalerator,IN_RBAI_Assistant Manager_Dispensing Process E...,**IN\_RBAI\_Assistant Manager\_Dispensing Proc...,en,"{""Website Domain"": ""bosch.com"", ""Job Opening U..."
1,https://jobs.smartrecruiters.com/BoschGroup/74...,job_ad,kaggle_techsalerator,Professional Internship: Hardware Development ...,**Professional Internship: Hardware Developmen...,en,"{""Website Domain"": ""bosch.com"", ""Job Opening U..."
2,https://jobs.zf.com/job/Shenyang-Process-Exper...,job_ad,kaggle_techsalerator,Process Expert BMS Production,ZF is a global technology company supplying sy...,en,"{""Website Domain"": ""zf.com"", ""Job Opening URL""..."


<class 'pandas.core.frame.DataFrame'>
Index: 9807 entries, 0 to 9918
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   doc_id         9807 non-null   object
 1   source_type    9807 non-null   object
 2   source_name    9807 non-null   object
 3   job_title_raw  9807 non-null   object
 4   raw_text       9807 non-null   object
 5   language       9807 non-null   object
 6   meta_json      9807 non-null   object
dtypes: object(7)
memory usage: 612.9+ KB


Save:

In [27]:
out_path = DATA_INTERIM_EXTERNAL / "techsalerator_job_postings_unified.parquet"

df_techsalerator_u.to_parquet(out_path, index=False)

print("Gespeichert unter:", out_path)
print("Anzahl vereinheitlichter Dokumente:", len(df_techsalerator_u))

Gespeichert unter: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim_external\techsalerator_job_postings_unified.parquet
Anzahl vereinheitlichter Dokumente: 9807


## 2.3 LinkedIn Job Postings Dataset (2023–2024), Largest File (postings.csv)

Source: Kaggle Gamil - LinkedIn Job Postings 2023-2024: https://www.kaggle.com/code/mahmoudredagamail/linkedin-job-postings-2023-2024?select=postings.csv; November 23, 2025

In [28]:
# Import
path_big = PATH_JOB_ADS / "postings.csv"

df_big_raw = pd.read_csv(
    path_big,
    encoding="utf-8",
    low_memory=False
)

In [29]:
# View File
display(df_big_raw.head(3))
print("Anzahl Anzeigen (raw):", len(df_big_raw))
df_big_raw.info()

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0


Anzahl Anzeigen (raw): 123849
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null

In [30]:
# Unified Document Schema, DataFrame
def unify_big(df):
    df_unified = pd.DataFrame({
        "doc_id": df["job_id"].astype(str),
        "source_type": "job_ad",
        "source_name": "kaggle_linkedin_2023_2024_big",
        "job_title_raw": df["title"],
        "raw_text": (
            df["title"].fillna("").astype(str) + "\n\n" +
            df["description"].fillna("").astype(str) + "\n\n" +
            df["skills_desc"].fillna("").astype(str) + "\n\n" +
            df["formatted_experience_level"].fillna("").astype(str)
        ).str.strip(),
        "language": "en",
        "meta_json": df.apply(
            lambda row: json.dumps({
                "job_id": row["job_id"],
                "company_id": row.get("company_id"),
                "location": row.get("location"),
                "formatted_work_type": row.get("formatted_work_type"),
                "skills_desc": row.get("skills_desc"),
                "pay_period": row.get("pay_period"),
            }),
            axis=1
        )
    })
    return df_unified

df_big_posts = unify_big(df_big_raw)

In [31]:
# Output of results
display(df_big_posts.head(3))
print("Anzahl vereinheitlicht:", len(df_big_posts))

,doc_id,source_type,source_name,job_title_raw,raw_text,language,meta_json
0,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,Marketing Coordinator\n\nJob descriptionA lead...,en,"{""job_id"": 921716, ""company_id"": 2774458.0, ""l..."
1,1829192,job_ad,kaggle_linkedin_2023_2024_big,Mental Health Therapist/Counselor,Mental Health Therapist/Counselor\n\nAt Aspen ...,en,"{""job_id"": 1829192, ""company_id"": NaN, ""locati..."
2,10998357,job_ad,kaggle_linkedin_2023_2024_big,Assitant Restaurant Manager,Assitant Restaurant Manager\n\nThe National Ex...,en,"{""job_id"": 10998357, ""company_id"": 64896719.0,..."


Anzahl vereinheitlicht: 123849


In [32]:
# Save
df_big_posts.to_parquet(
    DATA_INTERIM_EXTERNAL / "postings_big_unified.parquet",
    index=False
)

Data Understanding: LinkedIn postings.csv (large dataset)

- Rows: ~124k job postings
- Relevant fields: title (job title), description (full free text), location, skills_desc (partial skills)
- Evaluation criteria: Job title present, free text available for skill extraction, large sample size, cross-industry, entirely in English (US and CA)

# 3. Resumes/CVs/Applicant Profiles

Subfolder: cvs

## 3.1 Kaggle Resume Dataset (structured)

- Source: Kaggle Ganesh - 54k Resume Dataset (structured) consisting of multiple CSV files: https://www.kaggle.com/datasets/suriyaganesh/resume-dataset-structured/data?select=01_people.csv; 12/23/2025
- Free-text profiles for skill extraction
- Files in the Resume dataset (structured) subfolder: 01_people.csv, 02_abilities.csv, 03_education.csv, 04_experience.csv, 05_person_skills.csv, 06_skills.csv. Linked via person_id, must first be merged.
- Will be homogenized into the Unified Document Schema for `documents_raw.parquet`

In [33]:
# Paths + Loading
BASE = DATA_RAW_EXTERNAL / "cvs" / "Resume dataset (structured)"

people = pd.read_csv(BASE / "01_people.csv")
abilities = pd.read_csv(BASE / "02_abilities.csv")
edu = pd.read_csv(BASE / "03_education.csv")
exp = pd.read_csv(BASE / "04_experience.csv")
person_skills = pd.read_csv(BASE / "05_person_skills.csv")
# skills_master = pd.read_csv(BASE / "06_skills.csv")  # Not necessary for now; possibly in Skill Vocabulary

Minimal Cleaning, Whitespace:

In [34]:
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x if x else None

for df, cols in [ # extract the information that's important to us from all the CSV files
    (people, ["name"]),  # email/phone/linkedin ignore
    (abilities, ["ability"]),
    (edu, ["institution", "program", "location", "start_date"]),
    (exp, ["title", "location", "start_date", "end_date"]),  # company ignore
    (person_skills, ["skill"]),
]:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].apply(clean_text)

For data protection handling:

In [35]:
# Remove columns completely
for col in ["email", "phone", "linkedin"]:
    if col in people.columns:
        people = people.drop(columns=[col])
if "firm" in exp.columns:
    exp = exp.drop(columns=["firm"])

Removed fields sensitive to data protection (email/phone/LinkedIn); removed company name (not necessary for skill extraction)

Now, aggregation per person - a parameter that links the entries from all CSV files:

In [36]:
# Skills per Person
skills_agg = (
    person_skills.dropna(subset=["skill"])
    .groupby("person_id")["skill"]
    .apply(lambda s: sorted(set(s)))
)

# Skills per Person
abilities_agg = (
    abilities.dropna(subset=["ability"])
    .groupby("person_id")["ability"]
    .apply(lambda s: sorted(set(s)))
)

# Experience per Person
exp_titles_agg = (
    exp.dropna(subset=["title"])
    .groupby("person_id")["title"]
    .apply(lambda s: [t for t in list(dict.fromkeys(s))])  # unique, order-ish
)

# Most recent title, if start_date is parseable: sort; otherwise, use the first occurrence
most_title = exp.dropna(subset=["title"]).groupby("person_id")["title"].first()

# Education per Person
def edu_join(df):
    parts = []
    for _, r in df.iterrows():
        inst = r.get("institution")
        prog = r.get("program")
        sd = r.get("start_date")
        piece = " / ".join([p for p in [prog, inst, sd] if p])
        if piece:
            parts.append(piece)
    return parts

edu_agg = (
    edu.assign(_piece=edu.apply(
        lambda r: " / ".join([p for p in [r.get("program"), r.get("institution"), r.get("start_date")] if p]),
        axis=1
    ))
    .dropna(subset=["_piece"])
    .groupby("person_id")["_piece"]
    .apply(lambda s: list(dict.fromkeys(s)))  # unique, order-ish
)

Unified Document Schema:

In [37]:
def build_meta(pid):
    meta = {
        "n_skills": int(len(skills_agg.get(pid, []))) if pid in skills_agg.index else 0,
        "n_abilities": int(len(abilities_agg.get(pid, []))) if pid in abilities_agg.index else 0,
        "n_experience_titles": int(len(exp_titles_agg.get(pid, []))) if pid in exp_titles_agg.index else 0,
        "n_education": int(len(edu_agg.get(pid, []))) if pid in edu_agg.index else 0,
    }
    return json.dumps(meta, ensure_ascii=False)

def choose_job_title(pid, fallback_name):
    # most experience title
    t = most_title.get(pid, None) if pid in most_title.index else None
    if t:
        return t
    # fallback: people.name
    return fallback_name or ""

rows = []
for _, r in people.iterrows():
    pid = r["person_id"]
    fallback_name = r.get("name", None)

    job_title = choose_job_title(pid, fallback_name)

    skills = skills_agg.get(pid, []) if pid in skills_agg.index else []
    abils = abilities_agg.get(pid, []) if pid in abilities_agg.index else []
    expt = exp_titles_agg.get(pid, []) if pid in exp_titles_agg.index else []
    edul = edu_agg.get(pid, []) if pid in edu_agg.index else []

    raw_parts = []
    if job_title:
        raw_parts.append(f"TITLE: {job_title}")
    if skills:
        raw_parts.append("SKILLS: " + "; ".join(skills))
    if abils:
        raw_parts.append("ABILITIES: " + "; ".join(abils))
    if expt:
        raw_parts.append("EXPERIENCE_TITLES: " + "; ".join(expt))
    if edul:
        raw_parts.append("EDUCATION: " + "; ".join(edul))

    raw_text = "\n".join(raw_parts)
    raw_text = clean_text(raw_text)

    if not raw_text:
        continue

    rows.append({
        "doc_id": f"resume_struct_{pid}",
        "source_type": "cv",
        "source_name": "kaggle_resume_structured",
        "job_title_raw": clean_text(job_title),
        "raw_text": raw_text,
        "language": "en",
        "meta_json": build_meta(pid),
    })

df_resume_struct_u = pd.DataFrame(rows)
df_resume_struct_u.head(3)

,doc_id,source_type,source_name,job_title_raw,raw_text,language,meta_json
0,resume_struct_1,cv,kaggle_resume_structured,Database Administrator,TITLE: Database Administrator SKILLS: Backups;...,en,"{""n_skills"": 20, ""n_abilities"": 49, ""n_experie..."
1,resume_struct_2,cv,kaggle_resume_structured,Database Administrator,TITLE: Database Administrator SKILLS: assembly...,en,"{""n_skills"": 17, ""n_abilities"": 21, ""n_experie..."
2,resume_struct_3,cv,kaggle_resume_structured,Oracle Database Administrator,TITLE: Oracle Database Administrator SKILLS: D...,en,"{""n_skills"": 7, ""n_abilities"": 21, ""n_experien..."


Structure:
- raw_text: Sections TITLE/SKILLS/ABILITIES/EXPERIENCE_TITLES/EDUCATION
- Job title = an available experience title
- Fallback = people.name (since jobs are listed there instead of names)

Save:

In [38]:
out_path = DATA_INTERIM_EXTERNAL / "resume_structured_unified.parquet"
df_resume_struct_u.to_parquet(out_path, index=False)
print("Saved:", out_path, "rows:", len(df_resume_struct_u))

Saved: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim_external\resume_structured_unified.parquet rows: 54933


Sanity Checks:

In [39]:
print("people:", len(people))
print("unified:", len(df_resume_struct_u))
print("unique doc_id:", df_resume_struct_u["doc_id"].nunique())

df_resume_struct_u["raw_len"] = df_resume_struct_u["raw_text"].str.len()
df_resume_struct_u["raw_len"].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99])

people: 54933
unified: 54933
unique doc_id: 54933


count    54933.000000
mean      1612.046366
std       1304.818942
min         99.000000
10%        607.000000
25%        848.000000
50%       1224.000000
75%       1914.000000
90%       2997.000000
95%       3988.000000
99%       7048.000000
max      15150.000000
Name: raw_len, dtype: float64

Sanity:
- Number of people vs. number of unified documents equals 54,933
- Unique doc_id
- Text length distribution as a plausibility check

# 4. Social Media/LinkedIn-style Profiles & Skill Lists
Subfolder: social_media_data

## 4.1 Kaggle LinkedIn Profiles Morocco

Source: Kaggle Abbour - LinkedIn Profiles Morocco: https://www.kaggle.com/datasets/fatimazahraabbour1/linkedin-profiles-morocco?select=LinkedIn_Profiles_Morocco.csv; November 23, 2025

Supplement to job postings & CV data

Dataset containing profile attributes such as: headline, address, description, education, experience, certifications, languages, …

In [40]:
# Paths + Setup
PATH_LINKEDIN = DATA_RAW_EXTERNAL / "social_media_data" / "LinkedIn_Profiles_Morocco.csv"
DATA_INTERIM_EXTERNAL.mkdir(exist_ok=True)

Loading data: The dataset contains LinkedIn-style profile information from Morocco (headline/description/education/experience/skills in text format). The encoding is inconsistent in some places -> `errors="ignore"`.

In [41]:
df_li_raw = pd.read_csv(PATH_LINKEDIN,
    encoding="utf-8",
    # on_bad_lines="skip"
)

df_li_raw.head(3)
df_li_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1525 entries, 0 to 1524
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   headline               1523 non-null   object
 1   address                1520 non-null   object
 2   description            1428 non-null   object
 3   followers              1382 non-null   object
 4   educations             1525 non-null   object
 5   nbr of educations      1525 non-null   int64 
 6   experiences            1525 non-null   object
 7   nbr of experiences     1525 non-null   int64 
 8   certifications         1525 non-null   object
 9   nbr of certifications  1525 non-null   int64 
 10  projects               1525 non-null   object
 11  nbr of projects        1525 non-null   int64 
 12  languages              1525 non-null   object
 13  nbr of languages       1525 non-null   int64 
dtypes: int64(5), object(9)
memory usage: 166.9+ KB


In [42]:
# Overview: Structure & Data Quality
print("Anzahl LinkedIn-Profile:", len(df_li_raw))
df_li_raw.info()

Anzahl LinkedIn-Profile: 1525
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1525 entries, 0 to 1524
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   headline               1523 non-null   object
 1   address                1520 non-null   object
 2   description            1428 non-null   object
 3   followers              1382 non-null   object
 4   educations             1525 non-null   object
 5   nbr of educations      1525 non-null   int64 
 6   experiences            1525 non-null   object
 7   nbr of experiences     1525 non-null   int64 
 8   certifications         1525 non-null   object
 9   nbr of certifications  1525 non-null   int64 
 10  projects               1525 non-null   object
 11  nbr of projects        1525 non-null   int64 
 12  languages              1525 non-null   object
 13  nbr of languages       1525 non-null   int64 
dtypes: int64(5), object(9)
memory usage: 166.9

Data Understanding:
- 1,525 profiles
- 14 attributes
- Key text fields: `headline`, `description`, `experiences`
- Nested fields as strings (list formats)
- Languages: primarily en, but also fr + ar

Light Cleaning:
- Empty strings -> NaN
- Remove HTML/Unicode garbage (rough cleanup)
- Leave long list fields (`educations`, `experiences`, …) as strings
- No in-depth cleanup, as the unified schema only requires free text

In [43]:
df_li = df_li_raw.copy()

# Basic cleaning
df_li = df_li.replace({"­": "", "\u200b": "", "\n": " "}, regex=True)

# Replace missing description with ""
df_li["description"] = df_li["description"].fillna("")

Mapping to the Unified Schema:
- `doc_id` -> sequential index
- `source_type` -> `"linkedin_profile"`
- `source_name` -> `"kaggle_linkedin_morocco"`
- `job_title_raw` -> `headline`
- `raw_text` -> Combination of all relevant text fields (descriptions, education, experience, certifications)
- `meta_json` -> Additional information (address, followers, …)

In [44]:
# DataFrame, Unified
def unify_linkedin(df):
    df_unified = pd.DataFrame({
        "doc_id": df.index.astype(str),

        "source_type": "linkedin_profile",
        "source_name": "kaggle_linkedin_morocco",

        "job_title_raw": df["headline"].astype(str),

        # Combine important free-text fields
        "raw_text": (
            df["description"].astype(str) + " " +
            df["educations"].astype(str) + " " +
            df["experiences"].astype(str) + " " +
            df["certifications"].astype(str)
        ),

        "language": "en", # is set to "en" by default; more specific detection will be performed later using langdetect

        "meta_json": df.apply(
            lambda row: json.dumps({
                "address": row["address"],
                "followers": row["followers"],
                "languages": row["languages"],
            }),
            axis=1
        )
    })
    return df_unified

df_li_posts = unify_linkedin(df_li)
df_li_posts.head(3)

,doc_id,source_type,source_name,job_title_raw,raw_text,language,meta_json
0,0,linkedin_profile,kaggle_linkedin_morocco,Data Engineer @SBI,[] [] [],en,"{""address"": ""Prefecture of Casablanca, Casabla..."
1,1,linkedin_profile,kaggle_linkedin_morocco,A second-year master's student in Big Data and...,Ã‰tudiante en deuxiÃ¨me annÃ©e Master Big Data...,en,"{""address"": ""Casablanca, Casablanca-Settat, Mo..."
2,2,linkedin_profile,kaggle_linkedin_morocco,Senior IT Technician @Dell | Data Science Stud...,I'm a Senior IT technician at Dell Technologie...,en,"{""address"": ""Casablanca-Settat, Morocco"", ""fol..."


- Each entry represents a LinkedIn profile
- `job_title_raw`: headline (specific roles)
- `raw_text`: combination of description + education + experience + certifications
- Additional information in `meta_json`
- Language is currently set to "en"; not yet normalized (will be done globally later)

In [45]:
# Result
print("Anzahl Profile (unified):", len(df_li_posts))

Anzahl Profile (unified): 1525


In [46]:
# Save
df_li_posts.to_parquet(
    DATA_INTERIM_EXTERNAL / "linkedin_morocco_unified.parquet",
    index=False
)

1,525 LinkedIn profiles successfully unified. Saved as `linkedin_morocco_unified.parquet`

# 5. Continuing Education/Course Data

## 5.1 Udemy Course Dataset

- Subfolder: courses
- Source: Kaggle Bayir - https://www.kaggle.com/datasets/emrebayirr/udemy-course-dataset-categories-ratings-and-trends; November 23, 2025
- Columns: id, title, url, is_paid, instructor_names, category, headline, num_subscribers, rating, num_reviews, instructional_level, objectives, curriculum
- As an additional dataset, possibly for course/training recommendations; less suitable for profile expansion, as it primarily lacks job titles and descriptions

In [47]:
# Setup + Paths
PATH_UDEMY = DATA_RAW_EXTERNAL / "courses" / "udemy_courses.csv"
DATA_INTERIM_EXTERNAL.mkdir(exist_ok=True)

Loading dataset: Dataset is large = low_memory=False. Instructor names = personal data, i.e., DO NOT include. Free text for skill extraction = headline + objectives + curriculum

In [48]:
df_ud_raw = pd.read_csv(
    PATH_UDEMY,
    encoding="utf-8",
    low_memory=False,
    on_bad_lines="skip"
)

display(df_ud_raw.head(3))
df_ud_raw.info()
print("Anzahl Kurse:", len(df_ud_raw))

,id,title,url,is_paid,instructor_names,category,headline,num_subscribers,rating,num_reviews,instructional_level,objectives,curriculum
0,567828,The Complete Python Bootcamp From Zero to Hero...,https://www.udemy.com/course/complete-python-b...,True,"Jose Portilla, Pierian Training",Development,Learn Python like a Professional Start from t...,1976866,4.576494,521219,All Levels,You will learn how to leverage the power of Py...,"Course Overview, Auto-Welcome Message, Course ..."
1,1565838,The Complete 2024 Web Development Bootcamp,https://www.udemy.com/course/the-complete-web-...,True,"Dr. Angela Yu, Developer and Lead Instructor",Development,Become a Full-Stack Web Developer with just ON...,1362586,4.679065,409793,All Levels,Build 16 web development projects for your por...,"Front-End Web Development, What You'll Get in ..."
2,2776760,100 Days of Code: The Complete Python Pro Boot...,https://www.udemy.com/course/100-days-of-code/,True,"Dr. Angela Yu, Developer and Lead Instructor",Development,Master Python by building 100 projects in 100 ...,1417942,4.698768,331803,All Levels,You will master the Python programming languag...,Day 1 - Beginner - Working with Variables in P...


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98104 entries, 0 to 98103
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   98104 non-null  int64  
 1   title                98104 non-null  object 
 2   url                  98104 non-null  object 
 3   is_paid              98104 non-null  bool   
 4   instructor_names     98102 non-null  object 
 5   category             98104 non-null  object 
 6   headline             98104 non-null  object 
 7   num_subscribers      98104 non-null  int64  
 8   rating               98104 non-null  float64
 9   num_reviews          98104 non-null  int64  
 10  instructional_level  98104 non-null  object 
 11  objectives           98104 non-null  object 
 12  curriculum           98104 non-null  object 
dtypes: bool(1), float64(1), int64(3), object(8)
memory usage: 9.1+ MB
Anzahl Kurse: 98104


In [49]:
print(df_ud_raw.isna().sum())
df_ud_raw.describe(include="all")

id                     0
title                  0
url                    0
is_paid                0
instructor_names       2
category               0
headline               0
num_subscribers        0
rating                 0
num_reviews            0
instructional_level    0
objectives             0
curriculum             0
dtype: int64


,id,title,url,is_paid,instructor_names,category,headline,num_subscribers,rating,num_reviews,instructional_level,objectives,curriculum
count,9.810400e+04,98104,98104,98104,98102,98104,98104,9.810400e+04,98104.000000,98104.000000,98104,98104,98104
unique,NaN,97678,98104,2,36973,13,95854,NaN,NaN,NaN,4,96453,97698
top,NaN,Forex Ultimate Best Course Step by Step A-Z > ...,https://www.udemy.com/course/complete-python-b...,True,Dr. José Prabhu J,Development,Towards Excellence,NaN,NaN,NaN,All Levels,Assessment Preparing for exams at other certif...,"Practice Tests, Practice Test 1, Practice Test..."
freq,NaN,5,1,93607,289,9945,40,NaN,NaN,NaN,53354,25,58
mean,3.360106e+06,NaN,NaN,NaN,NaN,NaN,NaN,5.765752e+03,4.091346,533.024688,NaN,NaN,NaN
std,1.817398e+06,NaN,NaN,NaN,NaN,NaN,NaN,2.444085e+04,1.091701,4728.577300,NaN,NaN,NaN
min,2.762000e+03,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000,0.000000,NaN,NaN,NaN
25%,1.709971e+06,NaN,NaN,NaN,NaN,NaN,NaN,1.220000e+02,4.000639,12.000000,NaN,NaN,NaN
50%,3.568165e+06,NaN,NaN,NaN,NaN,NaN,NaN,7.540000e+02,4.389837,46.500000,NaN,NaN,NaN
75%,4.962858e+06,NaN,NaN,NaN,NaN,NaN,NaN,3.493250e+03,4.639675,165.000000,NaN,NaN,NaN


Data Understanding
- ~98k Udemy courses
- Relevant fields for skill extraction: title (possibly general context), headline (short description), objectives (learning outcomes = skills?), curriculum (detailed course content)
- Instructor_names -> personal data -> remove
- No standardized role names -> later, only text for skills

Light Cleaning:

In [50]:
df_ud = df_ud_raw.copy()

# Remove personal information
df_ud = df_ud.drop(columns=["instructor_names"], errors="ignore")

# fill in the missing text
for col in ["headline", "objectives", "curriculum"]:
    df_ud[col] = df_ud[col].fillna("")

# Generate free-form text
df_ud["raw_text"] = (
    df_ud["headline"].astype(str) + " " +
    df_ud["objectives"].astype(str) + " " +
    df_ud["curriculum"].astype(str)
)

Create a unified schema:

In [51]:
# Unified DataFrame
def unify_udemy(df):
    df_uni = pd.DataFrame({
        "doc_id": df["id"],
        "source_type": "course",
        "source_name": "kaggle_udemy_courses",
        
        # The role name does not exist, so = Title
        "job_title_raw": df["title"],
        
        # Free text
        "raw_text": df["raw_text"],
        "language": None, # Language (later via LangDetect)
        
        "meta_json": df.apply(
            lambda row: json.dumps({
                "title": row["title"],
                "category": row["category"],
                "is_paid": row["is_paid"],
                "rating": row["rating"],
                "num_subscribers": row["num_subscribers"],
                "num_reviews": row["num_reviews"],
                "instructional_level": row["instructional_level"]
            }),
            axis=1
        )
    })
    return df_uni

df_ud_posts = unify_udemy(df_ud)
df_ud_posts.head(3)
print("Anzahl Kurse (unified):", len(df_ud_posts))

Anzahl Kurse (unified): 98104


In [52]:
df_ud_posts.head(5)

,doc_id,source_type,source_name,job_title_raw,raw_text,language,meta_json
0,567828,course,kaggle_udemy_courses,The Complete Python Bootcamp From Zero to Hero...,Learn Python like a Professional Start from t...,None,"{""title"": ""The Complete Python Bootcamp From Z..."
1,1565838,course,kaggle_udemy_courses,The Complete 2024 Web Development Bootcamp,Become a Full-Stack Web Developer with just ON...,None,"{""title"": ""The Complete 2024 Web Development B..."
2,2776760,course,kaggle_udemy_courses,100 Days of Code: The Complete Python Pro Boot...,Master Python by building 100 projects in 100 ...,None,"{""title"": ""100 Days of Code: The Complete Pyth..."
3,625204,course,kaggle_udemy_courses,The Web Developer Bootcamp 2024,10 Hours of React just added. Become a Develop...,None,"{""title"": ""The Web Developer Bootcamp 2024"", ""..."
4,1362070,course,kaggle_udemy_courses,React - The Complete Guide 2024 (incl. Next.js...,Dive in and learn React.js from scratch! Learn...,None,"{""title"": ""React - The Complete Guide 2024 (in..."


In [53]:
# Save
df_ud_posts.to_parquet(
    DATA_INTERIM_EXTERNAL / "courses_udemy_unified.parquet",
    index=False
)

# Conclusion: Notebook 05

In this notebook, external, non-standardized datasets (job postings, resumes, social media profiles) were systematically analyzed and converted into a uniform document schema.

Although all datasets differ significantly in terms of origin, structure, and purpose, together they reflect the real-world heterogeneity of labor market-related data:
- CVs and social media profiles provide individual-level skill indicators
- Job postings reflect the labor market context
- Course data broadens the perspective on training opportunities

This standardization enables comparable further processing and forms the basis for subsequent profile expansion methods.